In [1]:
import pandas as pd
from pathlib import Path

In [2]:
PROJECT_DIR = Path("/home/sc4r3cr0w/Schreibtisch/Data Analytics/WBS_Final_Project")
RAW = PROJECT_DIR / "RAW"
CLEAN_DIR = PROJECT_DIR / "CLEAN"
CLEAN_DIR.mkdir(exist_ok=True)

POPULATION_DIR = RAW / "population"

print(POPULATION_DIR.exists(), len(list(POPULATION_DIR.glob('*'))))

True 1


In [3]:
df = pd.read_csv(
    POPULATION_DIR / "12411-01-01-4.csv",
    sep=";",
    encoding="cp1252",
    skiprows=5,
    header=None,
    dtype=str,
)
print(df.shape)
df.head(6)

(546, 32)


,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
0,NaN,NaN,Stichtag,Stichtag,Stichtag,Stichtag,Stichtag,Stichtag,Stichtag,Stichtag,...,Stichtag,Stichtag,Stichtag,Stichtag,Stichtag,Stichtag,Stichtag,Stichtag,Stichtag,Stichtag
1,NaN,NaN,31.12.2025,31.12.2025,31.12.2025,31.12.2024,31.12.2024,31.12.2024,31.12.2023,31.12.2023,...,31.12.2019,31.12.2018,31.12.2018,31.12.2018,31.12.2017,31.12.2017,31.12.2017,31.12.2016,31.12.2016,31.12.2016
2,NaN,NaN,Insgesamt,männlich,weiblich,Insgesamt,männlich,weiblich,Insgesamt,männlich,...,weiblich,Insgesamt,männlich,weiblich,Insgesamt,männlich,weiblich,Insgesamt,männlich,weiblich
3,DG,Deutschland,83467117,41184322,42282795,83577140,41241701,42335439,83456045,41161931,...,42129098,83019213,40966691,42052522,82792351,40843565,41948786,82521653,40697118,41824535
4,01,Schleswig-Holstein,2958519,1447377,1511142,2959517,1447902,1511615,2953202,1443888,...,1480890,2896712,1419457,1477255,2889821,1416535,1473286,2881926,1412665,1469261
5,01001,"Flensburg, kreisfreie Stadt",95568,47298,48270,96326,47651,48675,96431,47553,...,45260,89504,44599,44905,88519,44086,44433,87432,43617,43815


In [4]:
date_row = df.iloc[1]
type_row = df.iloc[2]

total_cols = []
for col in df.columns:
    if type_row[col] == "Insgesamt":
        total_cols.append(col)

for col in total_cols:
    print(col, date_row[col])

2 31.12.2025
5 31.12.2024
8 31.12.2023
11 31.12.2022
14 31.12.2021
17 31.12.2020
20 31.12.2019
23 31.12.2018
26 31.12.2017
29 31.12.2016


In [5]:
keys = df[0]

is_district = keys.str.match(r"^\d{5}$", na=False)
is_city_state = keys.isin(["02", "11"])

districts = df[is_district | is_city_state].copy()
districts[0] = districts[0].replace({"02": "02000", "11": "11000"})

print(len(districts))
districts[[0, 1]].head()

474


,0,1
5,01001,"Flensburg, kreisfreie Stadt"
6,01002,"Kiel, kreisfreie Stadt"
7,01003,"Lübeck, kreisfreie Stadt, Hansestadt"
8,01004,"Neumünster, kreisfreie Stadt"
9,01051,"Dithmarschen, Kreis"


In [6]:
frames = []

for col in total_cols:
    reference_year = int(date_row[col][-4:])
    frame = pd.DataFrame({
        "year": reference_year + 1,
        "district_key": districts[0],
        "district_name": districts[1].str.strip(),
        "population": districts[col],
    })
    frames.append(frame)

population = pd.concat(frames, ignore_index=True)
print(len(population))
population.head()

4740


,year,district_key,district_name,population
0,2026,01001,"Flensburg, kreisfreie Stadt",95568
1,2026,01002,"Kiel, kreisfreie Stadt",251842
2,2026,01003,"Lübeck, kreisfreie Stadt, Hansestadt",217061
3,2026,01004,"Neumünster, kreisfreie Stadt",80145
4,2026,01051,"Dithmarschen, Kreis",133405


In [7]:
population = population[population["population"] != "-"].copy()
population["population"] = population["population"].astype(int)

yearly = population.groupby("year").agg(
    districts=("district_key", "size"),
    population_total=("population", "sum"),
)
yearly

,districts,population_total
year,,
2017,401,82521653
2018,401,82792351
2019,401,83019213
2020,401,83166711
2021,401,83155031
2022,400,83237124
2023,400,83118501
2024,400,83456045
2025,400,83577140


In [8]:
population.to_csv(CLEAN_DIR / "population.csv", index=False)
print("Saved:", CLEAN_DIR / "population.csv", "| rows:", len(population))

Saved: /home/sc4r3cr0w/Schreibtisch/Data Analytics/WBS_Final_Project/CLEAN/population.csv | rows: 4005
